In [ ]:
# An Oracle for Shor's Algorithm

**Erdős Institute — Quantum Computing Bootcamp, Mini-project 1**

## 1. Introduction

Shor's factoring algorithm reduces integer factorization to the problem of finding the multiplicative order of an integer \(a\) modulo \(N\), where \(\gcd(a,N)=1\).

The quantum order-finding procedure uses controlled powers of the modular multiplication operator

\[
U_a|y\rangle =
\begin{cases}
|ay\bmod N\rangle, & 0\le y<N,\\
|y\rangle, & N\le y<2^n,
\end{cases}
\]

where \(n=\lceil\log_2N\rceil\).

The objective of this project is to construct a reversible controlled implementation of \(U_a\), satisfying

\[
|c\rangle|y\rangle|0\rangle_w
\longmapsto
|c\rangle|g_{a,N}(c,y)\rangle|0\rangle_w,
\]

where

\[
g_{a,N}(c,y)=
\begin{cases}
ay\bmod N,&c=1,\ y<N,\\
y,&\text{otherwise}.
\end{cases}
\]

We follow the modular arithmetic construction of Beauregard, *Circuit for Shor's Algorithm Using 2n+3 Qubits* (2003). The implementation proceeds through Fourier-space addition, modular addition, controlled modular multiplication, and finally an in-place multiplication oracle using modular inverses and uncomputation.

We focus on \(N=15\) and implement all multipliers \(a\in(\mathbb Z/15\mathbb Z)^\times\).

Our implementation also handles the out-of-range computational-basis state \(y=15\), which must remain unchanged.

**Reference:** Stéphane Beauregard, [Circuit for Shor's Algorithm Using 2n+3 Qubits](https://arxiv.org/abs/quant-ph/0205095).

In [ ]:
import numpy as np
import pytest

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.synthesis import synth_qft_full

from erdos_project_1.modular_adder import controlled_modular_add


@pytest.mark.parametrize("a", [0, 3, 9, 14])
def test_modular_adder(a):
    N = 15
    m = 5

    modular_adder = controlled_modular_add(a, N)
    qft = synth_qft_full(m, do_swaps=True)
    arithmetic = list(range(2, m + 2))

    for c1 in (0, 1):
        for c2 in (0, 1):
            for b in range(N):
                qc = QuantumCircuit(m + 3)

                if c1:
                    qc.x(0)
                if c2:
                    qc.x(1)

                for j in range(m):
                    if (b >> j) & 1:
                        qc.x(2 + j)

                qc.compose(qft, qubits=arithmetic, inplace=True)
                qc.compose(modular_adder, inplace=True)
                qc.compose(
                    qft.inverse(),
                    qubits=arithmetic,
                    inplace=True,
                )

                state = Statevector.from_instruction(qc)

                expected_b = (a + b) % N if c1 and c2 else b
                expected_index = (
                    c1 + 2 * c2 + 4 * expected_b
                )

                expected = np.zeros(2 ** (m + 3), dtype=complex)
                expected[expected_index] = 1

                assert np.allclose(state.data, expected, atol=1e-9)